In [1]:
from neo4j import GraphDatabase
from dotenv import load_dotenv
import json
import os
import sys
import numpy as np
import pandas as pd
import re
from py2neo import Graph
from IPython import InteractiveShell
InteractiveShell.ast_node_interactivity="all"

In [2]:
uri='neo4j://127.0.0.1:7687'
username='neo4j'
password='eKx2r34567'

In [3]:
def connect_to_neo4j():
    try:
        return Graph("bolt://localhost:7687", auth=("neo4j", "eKx2r34567"))
    except Exception as e:
        return None

graph = connect_to_neo4j()
graph.run("MATCH (n) RETURN n.name AS name, n.label AS label")

name,label
null,Monitoring
null,Monitoring
null,Monitoring


In [4]:
def load_data():
    query = """
        MATCH (n1)-[r1]-(c:Communication)-[r2]-(n2) WHERE r1.type="sent" AND r2.type="received"  
        RETURN n1.name as source_entity,r1.type as source_entity_action,c.content as content,c.timestamp as timestamp,r2.type as target_entity_action,n2.name as target_entity ORDER BY c.timestamp
    """
    df = graph.run(query).to_data_frame()
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['time'] = df['timestamp'].dt.time
    return df

df = load_data()

In [5]:
df['date']=df['timestamp'].dt.date
df['hour']=df['timestamp'].dt.hour
df['minute']=df['timestamp'].dt.minute
df['time']=df['timestamp'].dt.time
df['minutes_from_day_start']=df['timestamp'].dt.hour*60+df['timestamp'].dt.minute
df['minutes_from_day_start']=df['minutes_from_day_start'].astype(int)
df['minutes_from_day_start']=df['minutes_from_day_start'].astype(str)
df['minutes_from_day_start']=df['minutes_from_day_start'].astype(int)

In [6]:
df

,source_entity,source_entity_action,content,timestamp,target_entity_action,target_entity,date,hour,minute,time,minutes_from_day_start
0,The Lookout,sent,"Hey The Intern, it's The Lookout! Just spotted...",2040-10-01 08:09:00,received,The Intern,2040-10-01,8,9,08:09:00,489
1,The Intern,sent,"Hey The Lookout, The Intern here! I'd absolute...",2040-10-01 08:10:00,received,The Lookout,2040-10-01,8,10,08:10:00,490
2,Kelly,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",2040-10-01 08:13:00,received,Sam,2040-10-01,8,13,08:13:00,493
3,The Intern,sent,"Mrs. Money, it's The Intern. Just checking in ...",2040-10-01 08:16:00,received,Mrs. Money,2040-10-01,8,16,08:16:00,496
4,Mrs. Money,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",2040-10-01 08:19:00,received,Boss,2040-10-01,8,19,08:19:00,499
...,...,...,...,...,...,...,...,...,...,...,...
579,Oceanus City Council,sent,"Green Guardians, Oceanus City Council here. Yo...",2040-10-14 12:50:00,received,Green Guardians,2040-10-14,12,50,12:50:00,770
580,Green Guardians,sent,"EcoVigil, Green Guardians HQ here. Your video ...",2040-10-14 12:51:00,received,EcoVigil,2040-10-14,12,51,12:51:00,771
581,Defender,sent,Defender to Mako. Be advised that conservation...,2040-10-14 13:31:00,received,Mako,2040-10-14,13,31,13:31:00,811
582,Knowles,sent,"Knowles, Mako here. Proceed to southern dock a...",2040-10-14 13:34:00,received,Davis,2040-10-14,13,34,13:34:00,814


In [7]:
# Function to create dynamic time groups
def create_time_groups(timestamp, interval_minutes=15):
    """Convert timestamp to time group with specified interval"""
    hour = timestamp.hour
    minute = timestamp.minute
    
    # Find the start of the interval block
    start_minute = (minute // interval_minutes) * interval_minutes
    
    # Create the time group string with leading zero for single digits
    start_minute_str = f"{start_minute:02d}"
    
    # Calculate end minute
    end_minute = ((minute // interval_minutes) * interval_minutes) + interval_minutes
    if end_minute == 60:
        end_minute = 0
        hour += 1
    
    end_minute_str = f"{end_minute:02d}"
    
    # Format time with AM/PM
    ampm = 'am' if hour < 12 else 'pm'
    if hour > 12:
        hour -= 12
    elif hour == 0:
        hour = 12
    
    start_time = f"{hour}.{start_minute_str}{ampm}"
    
    # Handle end time AM/PM
    end_hour = timestamp.hour
    end_hour += ((minute // interval_minutes) * interval_minutes + interval_minutes) // 60
    end_ampm = 'am' if end_hour < 12 else 'pm'
    if end_hour > 12:
        end_hour -= 12
    elif end_hour == 0:
        end_hour = 12
    
    end_time = f"{end_hour}.{end_minute_str}{end_ampm}"
    
    return f"{start_time}-{end_time}"

# Apply the function to create time groups (default 15-minute intervals)
df['time_group'] = df['timestamp'].apply(lambda x: create_time_groups(x, 15))

# Also create a sortable version for chronological ordering
def create_sortable_time_group(timestamp, interval_minutes=15):
    """Create a sortable time group for chronological ordering"""
    hour = timestamp.hour
    minute = timestamp.minute
    start_minute = (minute // interval_minutes) * interval_minutes
    return hour * 60 + start_minute

df['time_group_sortable'] = df['timestamp'].apply(lambda x: create_sortable_time_group(x, 15))

In [8]:
df

,source_entity,source_entity_action,content,timestamp,target_entity_action,target_entity,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable
0,The Lookout,sent,"Hey The Intern, it's The Lookout! Just spotted...",2040-10-01 08:09:00,received,The Intern,2040-10-01,8,9,08:09:00,489,8.00am-8.15am,480
1,The Intern,sent,"Hey The Lookout, The Intern here! I'd absolute...",2040-10-01 08:10:00,received,The Lookout,2040-10-01,8,10,08:10:00,490,8.00am-8.15am,480
2,Kelly,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",2040-10-01 08:13:00,received,Sam,2040-10-01,8,13,08:13:00,493,8.00am-8.15am,480
3,The Intern,sent,"Mrs. Money, it's The Intern. Just checking in ...",2040-10-01 08:16:00,received,Mrs. Money,2040-10-01,8,16,08:16:00,496,8.15am-8.30am,495
4,Mrs. Money,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",2040-10-01 08:19:00,received,Boss,2040-10-01,8,19,08:19:00,499,8.15am-8.30am,495
...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,Oceanus City Council,sent,"Green Guardians, Oceanus City Council here. Yo...",2040-10-14 12:50:00,received,Green Guardians,2040-10-14,12,50,12:50:00,770,1.45pm-1.00pm,765
580,Green Guardians,sent,"EcoVigil, Green Guardians HQ here. Your video ...",2040-10-14 12:51:00,received,EcoVigil,2040-10-14,12,51,12:51:00,771,1.45pm-1.00pm,765
581,Defender,sent,Defender to Mako. Be advised that conservation...,2040-10-14 13:31:00,received,Mako,2040-10-14,13,31,13:31:00,811,1.30pm-1.45pm,810
582,Knowles,sent,"Knowles, Mako here. Proceed to southern dock a...",2040-10-14 13:34:00,received,Davis,2040-10-14,13,34,13:34:00,814,1.30pm-1.45pm,810


In [10]:
df['timestamp'].min(),df['timestamp'].max()

(Timestamp('2040-10-01 08:09:00'), Timestamp('2040-10-14 13:35:00'))

In [11]:
df.groupby('date')['hour'].agg(['min','max']).reset_index()

,date,min,max
0,2040-10-01,8,12
1,2040-10-02,8,13
2,2040-10-03,8,11
3,2040-10-04,8,13
4,2040-10-05,8,12
5,2040-10-06,9,13
6,2040-10-07,8,12
7,2040-10-08,8,11
8,2040-10-09,8,13
9,2040-10-10,8,14


In [12]:
df.to_csv("commsdata.csv",index=False)

In [13]:
df['source_entity'].value_counts(ascending=False)
df['target_entity'].value_counts(ascending=False)

source_entity
Green Guardians         44
Oceanus City Council    37
Mako                    35
Neptune                 34
Reef Guardian           34
The Lookout             33
Himark Harbor           27
Davis                   25
Remora                  25
The Intern              24
Clepper Jensen          20
Mrs. Money              19
Miranda Jordan          18
Sentinel                18
Paackland Harbor        16
V. Miesel Shipping      15
Rodriguez               14
Horizon                 14
EcoVigil                14
The Middleman           13
Liam Thorne             11
Boss                     9
Samantha Blake           9
Nadia Conti              8
Serenity                 8
Haacklee Harbor          7
Marlin                   7
Small Fry                7
Osprey                   6
Elise                    5
Defender                 5
Northern Light           4
Glitters Team            4
Knowles                  4
Seawatch                 3
The Accountant           3
Sailor Shifts 

target_entity
Mako                    58
Oceanus City Council    45
Remora                  37
Reef Guardian           28
Green Guardians         27
Neptune                 26
Mrs. Money              24
Sentinel                23
Horizon                 22
Boss                    21
Miranda Jordan          20
Nadia Conti             18
V. Miesel Shipping      18
Clepper Jensen          18
The Intern              18
Himark Harbor           18
Davis                   18
Liam Thorne             16
Paackland Harbor        16
EcoVigil                15
Seawatch                12
Sam                     10
Serenity                 8
Sailor Shifts Team       8
Marlin                   8
The Lookout              8
Samantha Blake           7
The Middleman            7
Rodriguez                6
Elise                    6
Northern Light           4
Osprey                   3
The Accountant           3
Defender                 2
Haacklee Harbor          2
City Officials           1
Kelly         

In [ ]:
df['date']=pd.to_datetime(df['date'])
df.loc[df['date']=='2040-10-01']

,source_entity,source_entity_action,content,timestamp,target_entity_action,target_entity,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable
0,The Lookout,sent,"Hey The Intern, it's The Lookout! Just spotted...",2040-10-01 08:09:00,received,The Intern,2040-10-01,8,9,08:09:00,489,8.00am-8.15am,480
1,The Intern,sent,"Hey The Lookout, The Intern here! I'd absolute...",2040-10-01 08:10:00,received,The Lookout,2040-10-01,8,10,08:10:00,490,8.00am-8.15am,480
2,Kelly,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",2040-10-01 08:13:00,received,Sam,2040-10-01,8,13,08:13:00,493,8.00am-8.15am,480
3,The Intern,sent,"Mrs. Money, it's The Intern. Just checking in ...",2040-10-01 08:16:00,received,Mrs. Money,2040-10-01,8,16,08:16:00,496,8.15am-8.30am,495
4,Mrs. Money,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",2040-10-01 08:19:00,received,Boss,2040-10-01,8,19,08:19:00,499,8.15am-8.30am,495
5,Boss,sent,"Mrs. Money, this is Boss. I'm available tomorr...",2040-10-01 08:21:00,received,Mrs. Money,2040-10-01,8,21,08:21:00,501,8.15am-8.30am,495
6,Mrs. Money,sent,"Boss, Mrs. Money here. I'll bring the updated ...",2040-10-01 08:24:00,received,Boss,2040-10-01,8,24,08:24:00,504,8.15am-8.30am,495
7,Boss,sent,"Middleman, this is Boss. I'd like to move our ...",2040-10-01 08:26:00,received,The Middleman,2040-10-01,8,26,08:26:00,506,8.15am-8.30am,495
8,The Middleman,sent,"Boss, this is The Middleman. I can meet earlie...",2040-10-01 08:29:00,received,Boss,2040-10-01,8,29,08:29:00,509,8.15am-8.30am,495
9,Boss,sent,"Middleman, this is Boss. Let's meet tomorrow a...",2040-10-01 08:32:00,received,The Middleman,2040-10-01,8,32,08:32:00,512,8.30am-8.45am,510


In [29]:
df.loc[df['hour']>=14]

,source_entity,source_entity_action,content,timestamp,target_entity_action,target_entity,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable
411,Reef Guardian,sent,"Seawatch, Reef Guardian here. Will commence ea...",2040-10-10 14:00:00,received,Seawatch,2040-10-10,14,0,14:00:00,840,2.00pm-2.15pm,840
412,The Lookout,sent,"Horizon, The Lookout here. Have started west q...",2040-10-10 14:03:00,received,Horizon,2040-10-10,14,3,14:03:00,843,2.00pm-2.15pm,840


In [25]:
df.dtypes

source_entity                     object
source_entity_action              object
content                           object
timestamp                 datetime64[ns]
target_entity_action              object
target_entity                     object
date                              object
hour                               int32
minute                             int32
time                              object
minutes_from_day_start             int64
time_group                        object
time_group_sortable                int64
dtype: object

In [16]:
df_cleaned=pd.read_csv("commsdata_cleaned.csv")

In [17]:
df_cleaned

,source_entity,source_entity_action,content,timestamp,target_entity_action,target_entity,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable
0,The Lookout,sent,"Hey The Intern, it's The Lookout! Just spotted...",01/10/40 8:09,received,The Intern,01/10/40,8,9,8:09:00,489,8.00am-8.15am,480
1,The Intern,sent,"Hey The Lookout, The Intern here! I'd absolute...",01/10/40 8:10,received,The Lookout,01/10/40,8,10,8:10:00,490,8.00am-8.15am,480
2,Kelly,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",01/10/40 8:13,received,Sam,01/10/40,8,13,8:13:00,493,8.00am-8.15am,480
3,The Intern,sent,"Mrs. Money, it's The Intern. Just checking in ...",01/10/40 8:16,received,Mrs. Money,01/10/40,8,16,8:16:00,496,8.15am-8.30am,495
4,Mrs. Money,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",01/10/40 8:19,received,Boss,01/10/40,8,19,8:19:00,499,8.15am-8.30am,495
...,...,...,...,...,...,...,...,...,...,...,...,...,...
575,Oceanus City Council,sent,"Green Guardians, Oceanus City Council here. Yo...",14/10/40 12:50,received,Green Guardians,14/10/40,12,50,12:50:00,770,1.45pm-1.00pm,765
576,Green Guardians,sent,"EcoVigil, Green Guardians HQ here. Your video ...",14/10/40 12:51,received,EcoVigil,14/10/40,12,51,12:51:00,771,1.45pm-1.00pm,765
577,Defender,sent,Defender to Mako. Be advised that conservation...,14/10/40 13:31,received,Mako,14/10/40,13,31,13:31:00,811,1.30pm-1.45pm,810
578,Mako,sent,"Knowles, Mako here. Proceed to southern dock a...",14/10/40 13:34,received,Knowles,14/10/40,13,34,13:34:00,814,1.30pm-1.45pm,810


In [3]:
def escape_cypher_identifier(identifier):
    """Escape a Cypher identifier by replacing special characters with underscores"""
    if not identifier:
        return "Node"
    # Replace spaces and special characters with underscores
    escaped = re.sub(r'[^a-zA-Z0-9_]', '_', str(identifier))
    # Ensure it starts with a letter or underscore
    if escaped and not escaped[0].isalpha() and escaped[0] != '_':
        pass
    # Ensure it's not empty
    if not escaped:
        escaped = "Node"
    return escaped

In [36]:
with open('MC3_schema.json', 'r', encoding='utf-8') as f:
    schema = json.load(f)
    schema=schema['schema']

In [37]:
with open('MC3_graph.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [39]:
entity_dict={}
for d in data['nodes']:
    if d['type']=='Entity':
        entity_dict[d['label']]=d['sub_type']

entity_dict

{'Sam': 'Person',
 'Kelly': 'Person',
 'Nadia Conti': 'Person',
 'Elise': 'Person',
 'Liam Thorne': 'Person',
 'Samantha Blake': 'Person',
 'Davis': 'Person',
 'Rodriguez': 'Person',
 'Sailor Shift': 'Person',
 'Clepper Jensen': 'Person',
 'Miranda Jordan': 'Person',
 'The Intern': 'Person',
 'The Lookout': 'Person',
 'The Accountant': 'Person',
 'Mrs. Money': 'Person',
 'The Middleman': 'Person',
 'Boss': 'Person',
 'Small Fry': 'Person',
 'Glitters Team': 'Organization',
 'Oceanus City Council': 'Organization',
 'Green Guardians': 'Organization',
 'V. Miesel Shipping': 'Organization',
 'Sailor Shifts Team': 'Organization',
 'Neptune': 'Vessel',
 'Marlin': 'Vessel',
 'Serenity': 'Vessel',
 'Mako': 'Vessel',
 'Reef Guardian': 'Vessel',
 'Horizon': 'Vessel',
 'Seawatch': 'Vessel',
 'EcoVigil': 'Vessel',
 'Sentinel': 'Vessel',
 'Osprey': 'Vessel',
 'Defender': 'Vessel',
 'Northern Light': 'Vessel',
 'Remora': 'Vessel',
 'Knowles': 'Vessel',
 "Mariner's Dream": 'Vessel',
 'Recreational Fi

In [41]:
df_cleaned['source_entity_type']=df_cleaned['source_entity'].map(entity_dict)
df_cleaned['target_entity_type']=df_cleaned['target_entity'].map(entity_dict)
df_cleaned['source_entity_type'].value_counts()
df_cleaned['target_entity_type'].value_counts()

source_entity_type
Vessel          247
Person          194
Organization     79
Location         60
Name: count, dtype: int64

target_entity_type
Vessel          241
Person          204
Organization     99
Location         36
Name: count, dtype: int64

In [48]:
df_cleaned.loc[:,['source_entity','source_entity_type','content','timestamp','target_entity','target_entity_type']].sample(5)

,source_entity,source_entity_type,content,timestamp,target_entity,target_entity_type
425,EcoVigil,Vessel,"Liam, this is EcoVigil. Per Council approval, ...",11/10/40 9:04,Liam Thorne,Person
472,Remora,Vessel,"Mako, Remora here. Stabilization equipment tes...",12/10/40 9:48,Mako,Vessel
447,Clepper Jensen,Person,"Miranda, Clepper here. Your discovery about fa...",11/10/40 10:16,Miranda Jordan,Person
27,Liam Thorne,Person,"Council, this is Liam Thorne. I've reviewed th...",01/10/40 10:56,Oceanus City Council,Organization
509,Remora,Vessel,"Davis, Remora here. Confirming night operation...",13/10/40 8:42,Davis,Person


In [53]:

df_cleaned['source']=df_cleaned['source_entity']
df_cleaned['target']=df_cleaned['target_entity']
df_cleaned=df_cleaned.drop(columns=['source_entity','target_entity'])
df_cleaned['timestamp'] = pd.to_datetime(df_cleaned['timestamp'])
df_cleaned['date'] = df_cleaned ['timestamp'].dt.date
df_cleaned['hour'] = df_cleaned['timestamp'].dt.hour
df_cleaned['time'] = df_cleaned['timestamp'].dt.time

# Calculate minutes since start of each day
df_cleaned['first_of_day'] = df_cleaned.groupby('date')['timestamp'].transform('min')
df_cleaned['mins_since_start_of_day'] = (df_cleaned['timestamp'] - df_cleaned['first_of_day']).dt.total_seconds() // 60
df_cleaned['mins_since_start_of_day'] = df_cleaned['mins_since_start_of_day'].astype(int)
df_cleaned = df_cleaned.drop(columns=['first_of_day'])

df_cleaned.head()

,source_entity_action,content,timestamp,target_entity_action,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable,source_entity_type,target_entity_type,mins_since_start_of_day,source,target
0,sent,"Hey The Intern, it's The Lookout! Just spotted...",2040-01-10 08:09:00,received,2040-01-10,8,9,08:09:00,489,8.00am-8.15am,480,Person,Person,0,The Lookout,The Intern
1,sent,"Hey The Lookout, The Intern here! I'd absolute...",2040-01-10 08:10:00,received,2040-01-10,8,10,08:10:00,490,8.00am-8.15am,480,Person,Person,1,The Intern,The Lookout
2,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",2040-01-10 08:13:00,received,2040-01-10,8,13,08:13:00,493,8.00am-8.15am,480,Person,Person,4,Kelly,Sam
3,sent,"Mrs. Money, it's The Intern. Just checking in ...",2040-01-10 08:16:00,received,2040-01-10,8,16,08:16:00,496,8.15am-8.30am,495,Person,Person,7,The Intern,Mrs. Money
4,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",2040-01-10 08:19:00,received,2040-01-10,8,19,08:19:00,499,8.15am-8.30am,495,Person,Person,10,Mrs. Money,Boss


In [55]:
df_cleaned.head()

,source_entity_action,content,timestamp,target_entity_action,date,hour,minute,time,minutes_from_day_start,time_group,time_group_sortable,source_entity_type,target_entity_type,mins_since_start_of_day,source,target
0,sent,"Hey The Intern, it's The Lookout! Just spotted...",2040-01-10 08:09:00,received,2040-01-10,8,9,08:09:00,489,8.00am-8.15am,480,Person,Person,0,The Lookout,The Intern
1,sent,"Hey The Lookout, The Intern here! I'd absolute...",2040-01-10 08:10:00,received,2040-01-10,8,10,08:10:00,490,8.00am-8.15am,480,Person,Person,1,The Intern,The Lookout
2,sent,"Sam, it's Kelly! Let's meet at Sunrise Point a...",2040-01-10 08:13:00,received,2040-01-10,8,13,08:13:00,493,8.00am-8.15am,480,Person,Person,4,Kelly,Sam
3,sent,"Mrs. Money, it's The Intern. Just checking in ...",2040-01-10 08:16:00,received,2040-01-10,8,16,08:16:00,496,8.15am-8.30am,495,Person,Person,7,The Intern,Mrs. Money
4,sent,"Boss, it's Mrs. Money. I've reviewed our opera...",2040-01-10 08:19:00,received,2040-01-10,8,19,08:19:00,499,8.15am-8.30am,495,Person,Person,10,Mrs. Money,Boss


In [54]:
df_cleaned.to_csv("MC3_comms_cleaned.csv",index=False)

In [38]:
entities=[d['label'] for d in data['nodes'] if d['type']=='Entity']

relationship_values=[d['label'] for d in data['nodes'] if d['type']=='Relationship']
event_values=[d['label'] for d in data['nodes'] if d['type']=='Event']
len(entities),len(relationship_values),len(event_values)

(72, 285, 802)

In [7]:
# --- Convert nodes ---
nodes_by_label = {}
node_type_schema = schema['nodes']
nodes_by_id = {}

In [8]:
list(node_type_schema.keys())

['Entity', 'Event', 'Relationship']

In [9]:
for node_key in list(node_type_schema.keys()):
    print(node_type_schema[node_key].keys())

dict_keys(['common_attributes', 'sub_types'])
dict_keys(['common_attributes', 'sub_types'])
dict_keys(['common_attributes', 'sub_types'])


In [10]:
def enrich_with_schema(obj, common_attrs, specific_attrs, obj_type, obj_id, context):
    enriched = dict(obj)
    # Add missing common attributes
    for k, v in common_attrs.items():
        if k not in enriched:
            enriched[k] = None
            #print(f"[WARN] {context} {obj_type} {obj_id} missing common attribute '{k}', set to None", file=sys.stderr)
    # Add missing specific attributes
    for k, v in specific_attrs.items():
        if k not in enriched:
            enriched[k] = None
            #print(f"[WARN] {context} {obj_type} {obj_id} missing specific attribute '{k}', set to None", file=sys.stderr)
    return enriched

In [11]:
for node in data['nodes']:
    node_label = node.get('label')
    node_type = node.get('type')
    node_sub_type=node.get('sub_type')
    node_name=node.get("name")
    node_copy = dict(node)
    # Use escaped ID as _uid to prevent Cypher syntax errors
    node_copy['id'] = escape_cypher_identifier(node['id'])
    nodes_by_id[node['id']] = node_copy
    # Schema enrichment with common and specific attributes
    if node_type in node_type_schema:
        type_info = node_type_schema[node_type]
        common_attrs = type_info.get('common_attributes', {})
        specific_attrs = {}
        if node_sub_type and 'sub_types' in type_info and node_sub_type in type_info['sub_types']:
                specific_attrs = type_info['sub_types'][node_sub_type].get('specific_attributes', {})
        node_copy = enrich_with_schema(node_copy, common_attrs, specific_attrs, node_label, node['id'], 'Node')
    else:
        pass
        #print(f"[WARN] Node type '{node_type}' not found in schema for node id {node['id']}", file=sys.stderr)
    nodes_by_label.setdefault(node_label, []).append(node_copy)

In [12]:
len(nodes_by_label.keys())

92

In [13]:
nodes_by_label.keys()

dict_keys(['Sam', 'Kelly', 'Nadia Conti', 'Elise', 'Liam Thorne', 'Samantha Blake', 'Davis', 'Rodriguez', 'Sailor Shift', 'Clepper Jensen', 'Miranda Jordan', 'The Intern', 'The Lookout', 'The Accountant', 'Mrs. Money', 'The Middleman', 'Boss', 'Small Fry', 'Glitters Team', 'Oceanus City Council', 'Green Guardians', 'V. Miesel Shipping', 'Sailor Shifts Team', 'Neptune', 'Marlin', 'Serenity', 'Mako', 'Reef Guardian', 'Horizon', 'Seawatch', 'EcoVigil', 'Sentinel', 'Osprey', 'Defender', 'Northern Light', 'Remora', 'Knowles', "Mariner's Dream", 'Recreational Fishing Boats', 'City Officials', 'Diving Tour Operators', 'Tourists', 'Conservation Vessels', 'Paackland Harbor', 'Haacklee Harbor', 'Himark Harbor', 'Nemo Reef', 'Restricted areas', 'Azure Cove', 'Protected areas', 'Eastern reefs', 'Eastern Coastline', 'Southern islands', 'Southern coastline', 'Coral Point', 'Dolphin Bay', 'E7', 'Northern quadrant', 'Southern quadrant', 'Eastern quadrant', 'Western quadrant', 'Eastern Islands', 'Weste

In [14]:
def upload_nodes_simple(nodes_data, driver):
    """Upload nodes using simple Cypher syntax"""
    # Create node label from type and sub_type
    try:
        node_type = nodes_data.get('type', 'Node')
        sub_type = nodes_data.get('sub_type', '')
        label = nodes_data.get('label')
        escaped_label = escape_cypher_identifier(label)
        
        # Create properties dict, excluding internal fields
        properties = {}

        for key, value in nodes_data.items():
            properties[key] = value
        
        # for key, value in nodes_data.items():
        #     # Convert dict or list of dicts to JSON string
        #     if isinstance(value, dict) or (isinstance(value, list) and any(isinstance(i, dict) for i in value)):
        #         properties[key] = json.dumps(value)
        #     else:
        #         properties[key] = value
            
        # Add _uid for identification
        properties['id'] = escape_cypher_identifier(nodes_data['id'])
    
        
        # Create Cypher query
        query = f"""
        MERGE (n:{escaped_label} {{id: $uid}})
        SET n += $properties
        RETURN n
        """
        
        driver.execute_query(query, uid=properties['id'], properties=properties)

    except Exception as ex:
        print(ex)

In [15]:
def convert_nodes_fixed(nodes: dict) -> tuple[str, dict]:
    """Fixed version of convert_nodes that properly escapes identifiers"""
    query = ""
    new_nodes = {}
    
    for node_label, node_data in nodes.items():
        # Escape the node label for safe use in Cypher
        escaped_label = escape_cypher_identifier(node_label)
        # Create a safe parameter key
        param_key = f"{escaped_label}"
        query += f"""CALL apoc.create.nodes(["{escaped_label}"], ${param_key});"""
        # Update the nodes dict with the new key
        new_nodes[param_key] = node_data
    
    return query, new_nodes

In [16]:
driver = GraphDatabase.driver(uri, auth=(username, password))
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [17]:
# d1=data['nodes'][:10]

In [18]:
nodes_by_id['Event_Assessment_14']

{'type': 'Event',
 'sub_type': 'Assessment',
 'label': 'Assessment',
 'timestamp': '2040-10-01 08:52:00',
 'assessment_type': 'documentation',
 'results': 'excellent conditions at Azure Cove',
 'id': 'Event_Assessment_14'}

In [19]:
# d=list(np.random.choice(data['nodes'],50,replace=False))

In [20]:
for k in (nodes_by_id.keys()):
    if nodes_by_id[k]['id']=='Event_TourActivity_137':
        nodes_data=nodes_by_id[k]
        upload_nodes_simple(nodes_data,driver)

In [21]:
edges = data.get('edges')

In [22]:
len(edges)

3226

In [23]:
if edges is None:
    for k, v in data.items():
        if k not in ('nodes', 'graph', 'directed', 'multigraph') and isinstance(v, list):
            edges = v
            break
    if edges is None:
        raise ValueError('Could not find edges/links in the JSON file.')

In [24]:
len(edges)

3226

In [25]:
rels_missing=[]
rels_missing_dict={"Event-Entity":"received","Entity-Relationship":"source","Entity-Event":"sent","Relationship-Entity":"target"}

In [26]:
def get_exact_relationship_type(edge, nodes_by_id, edge_types_schema):
    source_id = edge.get('source')
    target_id = edge.get('target')
    edge_type = edge.get('type',{})
    # Get source and target node types

    source_type = nodes_by_id.get(source_id, {}).get('type')
    target_type = nodes_by_id.get(target_id, {}).get('type')
    if edge_type=={}:
        #print(source_type,target_type,source_id,target_id)
        #rels_missing.append(source_type+"-"+target_type)
        edge_type=rels_missing_dict[source_type+"-"+target_type]
        edge['type']=edge_type

    # Try to match in schema
    for entry in edge_types_schema:
        if (
            entry.get('source') == source_type and
            entry.get('target') == target_type and
            (('type' not in entry and not edge_type) or (entry.get('type') == edge_type))
        ):
            return entry.get('type', edge_type or 'evidence_for')

In [27]:
rels_by_type = {}
rel_type_schema = schema['nodes'].get('Relationship', {}).get('sub_types', {})
edge_types_schema = schema.get('edges', {}).get('types', [])
for edge in edges:
    # Determine exact relationship type from schema
    rel_type = get_exact_relationship_type(edge, nodes_by_id, edge_types_schema)
    edge_copy = dict(edge)
    # Use escaped IDs for _from_uid and _to_uid to prevent Cypher syntax errors
    edge_copy['source'] = escape_cypher_identifier(edge['source'])
    edge_copy['target'] = escape_cypher_identifier(edge['target'])
    # Create a safe _uid for the relationship
    edge_id = edge.get('id', f"{edge['source']}_{rel_type}_{edge['target']}")
    edge_copy['id'] = escape_cypher_identifier(edge_id)
    # Schema enrichment for relationships (if type matches schema)
    if rel_type in rel_type_schema:
        common_attrs = schema['nodes']['Relationship'].get('common_attributes', {})
        specific_attrs = rel_type_schema[rel_type].get('specific_attributes', {})
        edge_copy = enrich_with_schema(edge_copy, common_attrs, specific_attrs, rel_type, edge_copy['_uid'], 'Edge')
    else:
        pass
        #print(f"[WARN] Edge type '{rel_type}' not found in schema for edge id {edge_copy['_uid']}", file=sys.stderr)
    rels_by_type.setdefault(rel_type, []).append(edge_copy)

In [30]:
# print(f"Uploading {len(rels_by_type)} relationships...")
success_count = 0
error_count = 0
for k in rels_by_type.keys():
    rel=rels_by_type[k]
    for r in rel:
        from_uid = escape_cypher_identifier(r['source'])
        to_uid = escape_cypher_identifier(r['target'])
        escaped_rel_type=r['type']
        properties = {}

        for key, value in r.items():
            properties[key] = value

            
        # for key, value in r.items():
        #     #if key not in ['source', 'target', 'type', 'id','_uid','is_inferred','_to_uid','_from_uid']:
        #     if key not in []:
        #         # Convert dict or list of dicts to JSON string
        #         if isinstance(value, dict) or (isinstance(value, list) and any(isinstance(i, dict) for i in value)):
        #             properties[key] = json.dumps(value)
        #         else:
                    #properties[key] = value
        
        query = f"""
            MATCH (a {{id: $from_uid}})
            MATCH (b {{id: $to_uid}})
            MERGE (a)-[r:{escaped_rel_type}]->(b)
            SET r += $properties
            RETURN r
            """

        driver.execute_query(query, from_uid=from_uid, to_uid=to_uid, properties=properties)
        success_count += 1



EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152929201188243684' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2276' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2348' labels=frozenset() properties={}>) type='source' properties={'is_inferred': True, 'id': '2', 'source': 'Sam', 'type': 'source', 'target': 'Relationship_Suspicious_217'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126ce9460>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1155181001001928932' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2276' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:940' labels=frozenset() properties={}>) type='source' properties={'is_inferred': True, 'id': '3013', 'source': 'Sam', 'type': 'source', 'target': 'Relationship_Colleagues_430'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x1034ff850>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:6917536724222477208' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2276' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:920' labels=frozenset() properties={}>) type='source' properties={'is_inferred': True, 'id': 'Sam_source_Relationship_Friends_272', 'source': 'Sam', 'type': 'source', 'target': 'Relationship_Friends_272'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126bb0970>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:6917536724222477221' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2276' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:933' labels=frozenset() properties={}>) type='source' properties={'is_inferred': True, 'id': 'Sam_source_Relationship_Colleagues_215', 'source': 'Sam', 'type': 'source', 'target': 'Relationship_Colleagues_215'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126cc5460>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:6917536724222477229' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2276' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:941' labels=frozenset() properties={}>) type='source' properties={'is_inferred': True, 'id': 'Sam_source_Relationship_Colleagues_431', 'source': 'Sam', 'type': 'source', 'target': 'Relationship_Colleagues_431'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x12690f6a0>, keys=['r'])

...

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152930300699871549' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2365' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2296' labels=frozenset() properties={}>) type='target' properties={'is_inferred': True, 'id': '2246', 'source': 'Relationship_Unfriendly_130', 'type': 'target', 'target': 'Green_Guardians'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126ec4f40>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152930300699871550' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2366' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2295' labels=frozenset() properties={}>) type='target' properties={'is_inferred': True, 'id': '2643', 'source': 'Relationship_Unfriendly_229', 'type': 'target', 'target': 'Oceanus_City_Council'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x12690fd00>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152930300699871551' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2367' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2302' labels=frozenset() properties={}>) type='target' properties={'is_inferred': True, 'id': '3014', 'source': 'Relationship_Unfriendly_312', 'type': 'target', 'target': 'Mako'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126e6ff40>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152930300699871552' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2368' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2295' labels=frozenset() properties={}>) type='target' properties={'is_inferred': True, 'id': '4034', 'source': 'Relationship_Unfriendly_567', 'type': 'target', 'target': 'Oceanus_City_Council'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126ec41f0>, keys=['r'])

EagerResult(records=[<Record r=<Relationship element_id='5:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:1152930300699871553' nodes=(<Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2369' labels=frozenset() properties={}>, <Node element_id='4:24d6b81b-a287-46ca-8ebe-dcb5619a7fcb:2296' labels=frozenset() properties={}>) type='target' properties={'is_inferred': True, 'id': '3992', 'source': 'Relationship_Unfriendly_625', 'type': 'target', 'target': 'Green_Guardians'}>>], summary=<neo4j._work.summary.ResultSummary object at 0x126e9bfa0>, keys=['r'])

In [31]:
print(f"Relationships uploaded: {success_count} successful, {error_count} failed")

Relationships uploaded: 3226 successful, 0 failed


In [ ]:
dsfsd

In [ ]:
def upload_relationships_simple(relationships_data, driver):
    """Upload relationships using simple Cypher syntax, one by one"""
    print(f"Uploading {len(relationships_data)} relationships...")
    
    success_count = 0
    error_count = 0

    for rel in relationships_data[:5]:
        try:
            rel_type = rel.get('type')
            if rel_type is None:
                source_type = rel.get('source_type')
                target_type = rel.get('target_type')
                print(source_type,target_type)
                rel_type=rels_missing_dict[source_type+"-"+target_type]
                rel['type']=rel_type

            escaped_rel_type = escape_cypher_identifier(rel_type)
            from_uid = escape_cypher_identifier(rel['source'])
            to_uid = escape_cypher_identifier(rel['target'])
            
            # Create properties dict, excluding internal fields
            properties = {}
            for key, value in rel.items():
                if key not in ['source', 'target', 'type', 'id']:
                    # Convert dict or list of dicts to JSON string
                    if isinstance(value, dict) or (isinstance(value, list) and any(isinstance(i, dict) for i in value)):
                        properties[key] = json.dumps(value)
                    else:
                        properties[key] = value
            
            # Create Cypher query using standard syntax
            query = f"""
            MATCH (a {{_uid: $from_uid}})
            MATCH (b {{_uid: $to_uid}})
            MERGE (a)-[r:{escaped_rel_type}]->(b)
            SET r += $properties
            RETURN r
            """
            
            #driver.execute_query(query, from_uid=from_uid, to_uid=to_uid, properties=properties)
            success_count += 1
            
        except Exception as e:
            rel_id = rel.get('id', f"{rel['source']}_{rel['target']}")
            print(f"Error uploading relationship {rel_id}: {e}")
            error_count += 1
    
    print(f"Relationships uploaded: {success_count} successful, {error_count} failed")


In [ ]:
for r in rels_by_type['source']:
    from_uid = escape_cypher_identifier(r['_from_uid'])
    to_uid = escape_cypher_identifier(r['_to_uid'])
    escaped_rel_type=r['type']
    properties = {}
    for key, value in r.items():
        if key not in ['source', 'target', 'type', 'id','_uid','is_inferred','_to_uid','__from_uid']:
            # Convert dict or list of dicts to JSON string
            if isinstance(value, dict) or (isinstance(value, list) and any(isinstance(i, dict) for i in value)):
                properties[key] = json.dumps(value)
            else:
                properties[key] = value
    query = f"""
        MATCH (a {{_uid: $from_uid}})
        MATCH (b {{_uid: $to_uid}})
        MERGE (a)-[r:{escaped_rel_type}]->(b)
        SET r += $properties
        RETURN r
        """

    with GraphDatabase.driver(uri, auth=(username, password)) as driver:
        driver.execute_query(query, from_uid=from_uid, to_uid=to_uid, properties=properties)

In [ ]:
rels_by_type.keys()

In [ ]:
neo4j_data = {
        'nodes': nodes_by_label,
        'relationships': rels_by_type
    }

In [ ]:
query, params = convert_nodes_fixed(neo4j_data['nodes'])

In [ ]:
params.keys()

In [ ]:
with GraphDatabase.driver(uri, auth=(username, password)) as driver:
  records, summary, keys = driver.execute_query(query, params)

In [ ]:
def convert_relationships_fixed(relationships: dict) -> (str, dict):
    """Fixed version of convert_relationships that properly escapes identifiers"""
    rel_record_list = []
    params = {}

    for rel_type, rel_records in relationships.items():
        # Escape the relationship type for safe use in Cypher
        escaped_rel_type = escape_cypher_identifier(rel_type)
        
        for record in rel_records:
            # Get the relationships _uid to distinguish it from other params
            record_key = record.get('_uid')
            # Create a safe parameter key
            safe_record_key = f"rel_{escape_cypher_identifier(record_key)}"
            params[safe_record_key] = record

            # Get the source and target node _uids (fixed field names)
            from_node_uid = record.get('_from_uid')
            to_node_uid = record.get('_to_uid')

            # Create a list of values and parameter keys which will be used to construct the relationship
            item = f"['{escaped_rel_type}', '{from_node_uid}', '{to_node_uid}', ${safe_record_key}]"
            rel_record_list.append(item)

    # Combine all the lists into a master list
    composite_rel_records_list = ",".join(rel_record_list)

    # Create a single query to process all the Relationship records
    query = f"""WITH [{composite_rel_records_list}] AS rel_data
                UNWIND rel_data AS relationship
                MATCH (n {{`_uid`:relationship[1]}})
                MATCH (n2 {{`_uid`:relationship[2]}})
                CALL apoc.create.relationship(n, relationship[0], relationship[3], n2) YIELD rel
                RETURN rel
    """

    return query, params

In [ ]:
rel_query, rel_params = convert_relationships_fixed(neo4j_data['relationships'])

In [ ]:
len(rel_params.keys())

In [ ]:
# Aggregate them into a single query and params dictionary
query += rel_query
params.update(rel_params)

In [ ]:
query

In [ ]:
with GraphDatabase.driver(uri, auth=(username, password)) as driver:
  records, summary, keys = driver.execute_query(query, params)